# CS 5326 – PA2: Kaggle model judging (Task 7)

Run this in a notebook created as a **utility benchmark** so `kaggle_benchmarks` gets free model-proxy access.
No GPU needed.

1. **Add Input** → the saved output of the training notebook (it contains `model-alignment/outputs/eval/*_pairs.jsonl`).
2. Optional: also attach a previous version of *this* notebook's output to resume judging.
3. **Run All**, then **Save Version** when it finishes.

It judges saved pair files only (it never regenerates outputs): 5 pairs per request, ≥ 5 s between attempts, 3 attempts per
batch, a judgment file rewritten after every batch, unresolved items logged to `*_failures.jsonl`, and per-comparison
win/loss/tie/unknown/failure summaries.

In [ ]:
REPO_URL = "https://github.com/alihasnaink/model-alignment.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/model-alignment"
JUDGE_MODEL = "google/gemini-3.6-flash"   # change if unavailable; the model used is recorded in every summary
PAIRS_INPUT = None   # None = auto-detect '*_pairs.jsonl' under /kaggle/input

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from IPython.display import FileLink, Image, Markdown, display

if not Path(REPO_DIR, "scripts").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
repo = Path(REPO_DIR)
os.chdir(repo)

import kaggle_benchmarks as kbench
assert kbench.kaggle.is_configured(), "No model-proxy credentials: create this notebook as a Kaggle utility benchmark."
print("Available judge models:", sorted(kbench.llms))
if JUDGE_MODEL not in kbench.llms:
    print(f"WARNING: {JUDGE_MODEL} not listed; the script will try to load it directly and fail clearly if it cannot.")

In [ ]:
# Gather the training notebook's outputs (pairs, generations, metrics, configs) into a writable directory.
if PAIRS_INPUT is None:
    found = sorted(Path("/kaggle/input").rglob("eval/*_pairs.jsonl"))
    assert found, "Attach the training notebook output (no eval/*_pairs.jsonl under /kaggle/input)."
    source_outputs = found[0].parent.parent
else:
    source_outputs = Path(PAIRS_INPUT)
OUT = Path("/kaggle/working/judge_outputs/outputs")
ignore = shutil.ignore_patterns("best", "final", "trainer", "*.safetensors", "*_train.jsonl", "*_valid.jsonl")
shutil.copytree(source_outputs, OUT, dirs_exist_ok=True, ignore=ignore)
# Resume from earlier judge runs if their outputs are attached too.
for previous in Path("/kaggle/input").rglob("eval/*_judged*.json*"):
    dest = OUT / "eval" / previous.name
    if not dest.exists():
        shutil.copy2(previous, dest)
EVAL = OUT / "eval"
print("Pair files:", [p.name for p in sorted(EVAL.glob("*_pairs.jsonl"))])

In [ ]:
env = {**os.environ, "PYTHONPATH": str(repo)}
subprocess.run([sys.executable, "scripts/run_judging.py", "--eval_dir", str(EVAL), "--judge_model", JUDGE_MODEL,
                "--backend", "kaggle", "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3"],
               check=True, env=env)

In [ ]:
for path in sorted(EVAL.glob("*_judged_summary.json")):
    s = json.loads(path.read_text())
    o = s["overall"]
    print(f"{path.stem:45s} {s['candidate']} vs {s['baseline']}: W {o['wins']} / T {o['ties']} / L {o['losses']} "
          f"/ unk {o['unknown']} / fail {o['failures']}  win-rate(excl ties)={o['win_rate_excluding_ties']}")
subprocess.run([sys.executable, "scripts/plot_metrics.py", "--outputs", str(OUT)], check=True, env=env)
subprocess.run([sys.executable, "scripts/summarize_results.py", "--outputs", str(OUT)], check=True, env=env)
if (OUT / "plots" / "judge_results.png").exists():
    display(Image(filename=str(OUT / "plots" / "judge_results.png")))

bundle = Path("/kaggle/working/judge_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            zf.write(path, Path("outputs") / path.relative_to(OUT))
os.chdir("/kaggle/working")
display(FileLink(bundle.name))
print("Done. Save Version to preserve the judgments.")